# Whisper vs FINAL benchmark (Colab, GPU)

3 Whisper models (base/small/medium) + your **FINAL** model (greedy and +KenLM), on the
same LibriSpeech dev-clean, under the same **clean / tel8k / noisy** conditions.
The degradation functions are identical to `ablation_engine`; ONE SHARED normalizer for all of
them; Whisper runs through torch-based HF so the GPU RAM figures are comparable.

**First:** Runtime > Change runtime type > **GPU**. Then run the cells in order.
Try `LIMIT=200` first, and if it looks right set `LIMIT=None` for the full run.

Measured: WER, CER, W/C, tel8k/clean ratio, RTF (compute/audio time), peak GPU RAM, wall time.


In [ ]:
# do NOT downgrade numpy (Colab pandas/datasets are built against numpy 2.x). pyctcdecode
# pins numpy<2 but works on 2.x, so install it with --no-deps and leave numpy alone
!pip install -q jiwer peft pygtrie https://github.com/kpu/kenlm/archive/master.zip
!pip install -q --no-deps pyctcdecode
# Colab's old torchao (0.10) breaks peft's LoRA dispatch; we do not use it.
!pip uninstall -y -q torchao


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, torch
print("GPU:", torch.cuda.get_device_name(0), "| bf16 support:", torch.cuda.is_bf16_supported())
FINAL_DIR = "/content/drive/MyDrive/CLEAR/Phase 1/runs/FINAL"
WHISPER   = ["openai/whisper-base", "openai/whisper-small", "openai/whisper-medium"]
CONDS     = ["clean", "tel8k", "noisy"]
LIMIT     = 200          # try small first, None means the full 2703
KA, KB    = 0.7, 0.5     # best KenLM alpha/beta (from the grid)
print("FINAL head.pt exists:", os.path.exists(FINAL_DIR + "/head.pt"))
print("FINAL adapter.pt exists:", os.path.exists(FINAL_DIR + "/adapter.pt"))


In [ ]:
from datasets import load_dataset
import numpy as np

def load_devclean(limit=None):
    ds = load_dataset("openslr/librispeech_asr", "clean", split="validation", streaming=True)
    clips = []
    for i, row in enumerate(ds):
        if limit and i >= limit: break
        a = row["audio"]; w = np.asarray(a["array"], np.float32); sr = a["sampling_rate"]
        if sr != 16000:
            w = np.interp(np.linspace(0, len(w)-1, int(len(w)*16000/sr)),
                          np.arange(len(w)), w).astype(np.float32)
        clips.append((w, row["text"]))
    return clips

CLIPS = load_devclean(LIMIT)
REFS = [t for _, t in CLIPS]
TOTAL_SEC = sum(len(w) for w, _ in CLIPS) / 16000
print(f"{len(CLIPS)} utt . total audio {TOTAL_SEC/60:.1f} min")


In [ ]:
import numpy as np, math
# IDENTICAL to degrade_eval in ablation_engine

def _mask(w, sr, lo, hi):
    if len(w) < 8: return w
    W = np.fft.rfft(w); f = np.fft.rfftfreq(len(w), 1.0/sr)
    if lo is not None: W[f < lo] = 0.0
    if hi is not None: W[f > hi] = 0.0
    return np.fft.irfft(W, len(w)).astype(np.float32)

def aug_band(w, sr=16000): return _mask(w, sr, 300.0, 3400.0)

def aug_8k(w, sr=16000):
    if len(w) < 8: return w
    x = _mask(w, sr, None, 3800.0); n8 = max(2, len(x)//2)
    d = np.interp(np.linspace(0, len(x)-1, n8), np.arange(len(x)), x)
    return np.interp(np.linspace(0, n8-1, len(x)), np.arange(n8), d).astype(np.float32)

def _pink(n, rng):
    m = n//2 + 1
    s = (rng.standard_normal(m) + 1j*rng.standard_normal(m)).astype(np.complex64)
    f = np.arange(m, dtype=np.float32); f[0] = 1.0
    x = np.fft.irfft(s/np.sqrt(f), n).astype(np.float32)
    return x / (float(np.sqrt((x**2).mean())) + 1e-9)

def _mix(w, nz, snr_db):
    nz = nz / (float(np.sqrt((nz**2).mean())) + 1e-9)
    k = math.sqrt((float((w**2).mean()) + 1e-12) / (10.0**(snr_db/10.0)))
    return (w + k*nz).astype(np.float32)

def degrade(w, mode):
    if mode in (None, "clean"): return w
    if mode == "tel8k": return aug_8k(aug_band(w))
    if mode == "noisy": return _mix(w, _pink(len(w), np.random.default_rng(12345)), 10.0)
    raise ValueError(mode)


In [ ]:
import re, jiwer
# a SHARED normalizer for the reference and every hypothesis, for a fair comparison
_N = re.compile(r"[^A-Z' ]+")
def norm(s):
    s = s.upper().replace("|", " ")
    return " ".join(_N.sub(" ", s).split())

def score(refs, hyps):
    R = [norm(r) for r in refs]; H = [norm(h) for h in hyps]
    w = jiwer.wer(R, H); c = jiwer.cer(R, H)
    return {"wer": w, "cer": c, "wc": (w/c if c else 0.0)}

def log_softmax(l):
    m = l.max(-1, keepdims=True); e = np.exp(l - m)
    return (l - m) - np.log(e.sum(-1, keepdims=True))

RESULTS = {}


In [ ]:
import torch, time
from transformers import WhisperProcessor, WhisperForConditionalGeneration

def bench_whisper(name):
    proc = WhisperProcessor.from_pretrained(name)
    model = WhisperForConditionalGeneration.from_pretrained(
        name, torch_dtype=torch.float16).to("cuda").eval()
    per = {}
    for cond in CONDS:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        hyps = []; t0 = time.perf_counter()
        with torch.no_grad():
            for w, _ in CLIPS:
                x = degrade(w, cond)
                feat = proc(x, sampling_rate=16000, return_tensors="pt"
                            ).input_features.to("cuda", torch.float16)
                gen = model.generate(feat, language="en", task="transcribe",
                                     max_new_tokens=220)
                hyps.append(proc.batch_decode(gen, skip_special_tokens=True)[0])
        dt = time.perf_counter() - t0
        m = score(REFS, hyps)
        m.update(rtf=dt/TOTAL_SEC, sec=dt, gb=torch.cuda.max_memory_allocated()/1e9)
        per[cond] = m
        print(f"  [{name.split('/')[-1]:14s} {cond:6s}] WER {m['wer']*100:5.2f} "
              f"CER {m['cer']*100:5.2f} W/C {m['wc']:.2f} RTF {m['rtf']:.3f} "
              f"{m['gb']:.1f}GB {dt:.0f}s")
    del model; torch.cuda.empty_cache()
    return per

for _m in WHISPER:
    RESULTS[_m.split("/")[-1]] = bench_whisper(_m)


In [ ]:
import torch, torch.nn as nn, time, json, os, urllib.request, gzip
from itertools import groupby
from peft import LoraConfig, inject_adapter_in_model
from transformers import HubertModel
from pyctcdecode import build_ctcdecoder

CHARS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ'")
def build_vocab():
    v = {c: i for i, c in enumerate(CHARS)}
    v["|"], v["[UNK]"], v["[PAD]"] = len(v), len(v)+1, len(v)+2
    return v
VOCAB = build_vocab(); BLANK, UNK = VOCAB["[PAD]"], VOCAB["[UNK]"]
I2C = {i: c for c, i in VOCAB.items()}

def ctc_greedy(logits):
    ids = logits.argmax(-1)
    return "".join(I2C[k] for k, _ in groupby(ids.tolist())
                   if k not in (BLANK, UNK)).replace("|", " ").strip()

# --- load the FINAL model (mHuBERT-147 + LoRA + head), identical to ablation_engine ---
raw = json.load(open(FINAL_DIR + "/config.json"))
WS = tuple(sorted(int(x) for x in raw.get("ws", (9, 10, 11, 12))))
LL = raw.get("lora_layers") or list(range(1, max(WS)+1))
HID = int(raw.get("hid", 768)); dev = "cuda"
AMP = torch.float16   # the T4 uses fp16 tensor cores (not bf16); Whisper is fp16 too -> fair
bb = HubertModel.from_pretrained("utter-project/mHuBERT-147",
                                 attn_implementation="sdpa",
                                 torch_dtype=torch.float16).to(dev)
lc = LoraConfig(r=int(raw.get("lora_r", 16)), lora_alpha=int(raw.get("lora_alpha", 32)),
                target_modules=["q_proj", "v_proj"], bias="none",
                layers_to_transform=[i-1 for i in LL])
bb = inject_adapter_in_model(lc, bb)
bb.load_state_dict(torch.load(FINAL_DIR + "/adapter.pt", map_location=dev), strict=False)
bb.eval()

class Head(nn.Module):
    def __init__(s, n, d, V):
        super().__init__(); s.layer_w = nn.Parameter(torch.zeros(n))
        s.net = nn.Sequential(nn.Linear(d, d), nn.ELU(), nn.Dropout(0.0), nn.Linear(d, V))
    def forward(s, x):
        w = s.layer_w.softmax(0); return s.net((x * w[None, None, :, None]).sum(2))

head = Head(len(WS), HID, len(VOCAB)).to(dev)
sd = torch.load(FINAL_DIR + "/head.pt", map_location=dev)
if "net.2.weight" in sd and sd["net.2.weight"].shape[0] != sd["net.0.weight"].shape[0]:
    sd["net.3.weight"] = sd.pop("net.2.weight"); sd["net.3.bias"] = sd.pop("net.2.bias")
head.load_state_dict(sd, strict=False); head.eval()
flen = bb._get_feat_extract_output_lengths

# --- KenLM decoder (alpha/beta from the grid) ---
labels = [""] * len(VOCAB)
for t, i in VOCAB.items():
    labels[i] = " " if t == "|" else ("" if t == "[PAD]" else ("⁇" if t == "[UNK]" else t))
LMP = "/content/3-gram.pruned.1e-7.arpa"
if not os.path.exists(LMP):
    urllib.request.urlretrieve(
        "https://www.openslr.org/resources/11/3-gram.pruned.1e-7.arpa.gz", "/content/lm.gz")
    open(LMP, "wb").write(gzip.open("/content/lm.gz", "rb").read())
kdec = build_ctcdecoder(labels, kenlm_model_path=LMP, alpha=KA, beta=KB)

def bench_ours():
    g_per, k_per = {}, {}
    for cond in CONDS:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        logits_all = []; t0 = time.perf_counter()
        with torch.no_grad():
            for w, _ in CLIPS:
                x = torch.from_numpy(degrade(w, cond).astype(np.float32))[None].to(dev)
                with torch.autocast("cuda", dtype=AMP):
                    o = bb(x, output_hidden_states=True)
                    hs = torch.stack([o.hidden_states[L] for L in WS], 2)
                lg = head(hs.float())[0]
                xl = int(flen(torch.tensor([x.shape[1]], device=dev))[0])
                logits_all.append(lg[:xl].float().cpu().numpy())
        fwd = time.perf_counter() - t0
        gb = torch.cuda.max_memory_allocated() / 1e9
        gm = score(REFS, [ctc_greedy(l) for l in logits_all])
        gm.update(rtf=fwd/TOTAL_SEC, sec=fwd, gb=gb); g_per[cond] = gm
        t1 = time.perf_counter()
        k_hyps = [kdec.decode(log_softmax(l)) for l in logits_all]
        kt = time.perf_counter() - t1
        km = score(REFS, k_hyps)
        km.update(rtf=(fwd+kt)/TOTAL_SEC, sec=fwd+kt, gb=gb); k_per[cond] = km
        print(f"  [FINAL {cond:6s}] greedy WER {gm['wer']*100:5.2f} | "
              f"+KenLM WER {km['wer']*100:5.2f} CER {km['cer']*100:5.2f} "
              f"W/C {km['wc']:.2f} | fwd {fwd:.0f}s +lm {kt:.0f}s {gb:.1f}GB")
    return g_per, k_per

_g, _k = bench_ours()
RESULTS["FINAL (greedy)"] = _g
RESULTS["FINAL (+KenLM)"] = _k


In [ ]:
import pandas as pd, numpy as np
order = [w.split("/")[-1] for w in WHISPER] + ["FINAL (greedy)", "FINAL (+KenLM)"]
rows = []
for name in order:
    r = RESULTS[name]
    rows.append({
        "model": name,
        "clean WER": round(r["clean"]["wer"]*100, 2),
        "tel8k WER": round(r["tel8k"]["wer"]*100, 2),
        "noisy WER": round(r["noisy"]["wer"]*100, 2),
        "clean CER": round(r["clean"]["cer"]*100, 2),
        "tel/clean": round(r["tel8k"]["wer"]/r["clean"]["wer"], 2),
        "W/C(clean)": round(r["clean"]["wc"], 2),
        "RTF": round(float(np.mean([r[c]["rtf"] for c in CONDS])), 3),
        "GPU GB": round(max(r[c]["gb"] for c in CONDS), 1),
        "sec(3cond)": int(round(sum(r[c]["sec"] for c in CONDS))),
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
out = "/content/drive/MyDrive/CLEAR/Phase 1/runs/whisper_vs_final.csv"
df.to_csv(out, index=False)
print("\nCSV -> " + out)
